# Plant Disease Detection Training Script (EfficientNetB0)

Train the deep learning model directly from Colab, fully supporting resume on disconnect!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================
# DATASET SETUP
# ==============================
import os
DATASET_ZIP = '/content/drive/MyDrive/archiveraak.zip'

if not os.path.exists('/content/dataset'):
    print("Extracting dataset... (This might take a minute)")
    os.system(f'unzip -q "{DATASET_ZIP}" -d /content/dataset')
else:
    print("Dataset already extracted!")

In [ ]:
# ==============================
# IMPORT LIBRARIES & CONFIGURATION
# ==============================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os
import glob

# Auto-detect paths in case of variable extraction structures
train_paths = glob.glob('/content/dataset/**/train', recursive=True)
if not train_paths: 
    raise ValueError("Could not find 'train' directory!")

TRAIN_DIR = train_paths[0]

# Sometimes 'valid' is extracted to a completely different subfolder
valid_paths = glob.glob('/content/dataset/**/valid', recursive=True)
if not valid_paths:
    valid_paths = glob.glob('/content/dataset/**/val', recursive=True)

if not valid_paths:
    raise ValueError("Could not find 'valid' or 'val' directory!")
    
VALID_DIR = valid_paths[0]

# ----------------- RESUME SETUP -----------------
# Save the model checkpoint directly to Google Drive!
MODEL_SAVE_DIR = "/content/drive/MyDrive/plant_disease_model/"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, "plant_disease_efficientnet.h5")
TFLITE_SAVE_PATH = "model.tflite"

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 20

In [ ]:
# ==============================
# LABEL CLEANING
# ==============================
def clean_label(label):
    return label.replace("___", " ").replace("_", " ").lower()

# ==============================
# DATA GENERATORS
# ==============================
print("Initializing Data Generators...")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# ==============================
# SAVE LABELS
# ==============================
labels = sorted(train_generator.class_indices.keys())

with open("labels.txt", "w") as f:
    for label in labels:
        f.write(clean_label(label) + "\n")
    f.write("background\n")

print("Labels saved.")

In [ ]:
# ==============================
# BUILD OR LOAD MODEL
# ==============================
if os.path.exists(MODEL_SAVE_PATH):
    print(f"\n>>> FOUND EXISTING CHECKPOINT AT {MODEL_SAVE_PATH} <<<")
    print(">>> Loading the model to resume training... <<<")
    model = load_model(MODEL_SAVE_PATH)
else:
    print("\n>>> Building EfficientNet-B0 Model from scratch... <<<")
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    predictions = Dense(len(labels), activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    # Freeze base model
    for layer in base_model.layers:
        layer.trainable = False

    # Compile model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

# ==============================
# CALLBACKS
# ==============================
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True)
]

# ==============================
# INITIAL TRAINING
# ==============================
if not os.path.exists(MODEL_SAVE_PATH):
    print("Starting Phase 1: Initial Training...")
    model.fit(
        train_generator,
        epochs=5,
        validation_data=valid_generator,
        callbacks=callbacks
    )
else:
    print("\nPhase 1 Skipped! (Loaded existing trained weights)")

In [ ]:
# ==============================
# FINE TUNING
# ==============================
print("Starting/Resuming Phase 2: Fine-tuning...")
print("(Note: Epochs will show starting from 1, but your loaded weights are preserved!)")

# Ensure top layers are un-frozen regardless of whether we just loaded or finished Phase 1
for layer in model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=valid_generator,
    callbacks=callbacks
)

# ==============================
# CONVERT TO TFLITE
# ==============================
print("Converting finished model to TFLite...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open(TFLITE_SAVE_PATH, 'wb') as f:
    f.write(tflite_model)

print("TFLite model ready.")

In [ ]:
# ==============================
# DOWNLOAD FILES
# ==============================
from google.colab import files

files.download('model.tflite')
files.download('labels.txt')